# 02 — Parameter Count and Memory

> Companion to **Week 11**, Part 2 of the lecture.

## Why this matters

A neural network is just a big bag of numbers (its **parameters**). The total
memory it takes is roughly:

```
size = number_of_parameters × bytes_per_parameter
```

So once you know how many parameters a model has and how many bytes you store
each one in (FP32 = 4, FP16 = 2, INT8 = 1), you can estimate its size.


## Step 1 — Build three small models

We will define three tiny MLPs of growing size and count their parameters.


In [ ]:
import pandas as pd
import torch
import torch.nn as nn


def count_params(model):
    return sum(p.numel() for p in model.parameters())


small = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10),
)

medium = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

large = nn.Sequential(
    nn.Linear(64, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)

for name, model in [("small", small), ("medium", medium), ("large", large)]:
    print(f"{name:>6}: {count_params(model):>7,} parameters")


## Step 2 — Convert parameter counts to memory

Each parameter takes 4 bytes in FP32, 2 in FP16, 1 in INT8. We will convert
to kilobytes (KB) so the numbers are easy to read.


In [ ]:
rows = []
for name, model in [("small", small), ("medium", medium), ("large", large)]:
    params = count_params(model)
    rows.append(
        {
            "model":   name,
            "params":  params,
            "fp32_kb": params * 4 / 1024,
            "fp16_kb": params * 2 / 1024,
            "int8_kb": params * 1 / 1024,
        }
    )

pd.DataFrame(rows)


### What to notice

- The **large** model has way more parameters than the small one.
- For the same model, **INT8 is 4× smaller** than FP32 — same model, fewer
  bits per number.
- These savings scale: a 7-billion-parameter LLM is 28 GB in FP32 but only
  ~7 GB in INT8 (and ~3.5 GB in INT4!).

## 🧪 Your turn

Add a fourth model called `huge` with three hidden layers of 1024 units.
- How many parameters does it have?
- How big would it be in FP32? In INT8?
- Could it fit in your laptop's RAM (assume 8 GB)?
